# Classification Baseline — Multi-Model Comparison

Compares **LogisticRegression**, **RandomForest**, and **HistGradientBoosting** on outcome prediction.
Binary label: 1 = loan approved (any offer with `Accepted == True` in the trace), 0 = not approved.
Uses only log-based features (`BasicControlFlowFeatures`). Time-series features are excluded.

Checkpoints for the cleaned log and feature matrix use `_clf` suffixed base names so they coexist
with regression checkpoints in the same `CHECKPOINT_DIR`.

In [1]:
import contextlib
import logging
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from tqdm import tqdm

from spi_time_series.pipeline.checkpointing import checkpoint
from spi_time_series.models.trainer import train
from spi_time_series.data import Dataset
from spi_time_series.data.constants import VALID_END_ACTIVITIES
from spi_time_series.data.schemas import EvaluationReport, EventLog, ModelArtifact, PreprocessedData, RawData
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.preprocessing.preprocess import (
    _build_trace_samples,
    clean_data,
    sliding_window_factory,
    split_data,
)

logging.basicConfig(level=logging.INFO, force=True)

## Config

All tunable values live in the cells below.

In [2]:
RANDOM_STATE = 42
CV_FOLDS = 3
N_ITER = 10
MIN_PREFIX_LENGTH = 3
# Cap each trace at its MAX_PREFIX_LENGTH most recent events. Without a cap, BPI 2017
# generates ~800K training prefixes (avg trace length ~38, min_length=3). A cap of 10
# reduces this to ~220K and cuts extraction time proportionally.
MAX_PREFIX_LENGTH = 10

CHECKPOINT_DIR = Path("../data/checkpoints")

In [3]:
MODEL_CONFIGS = {
    "logistic_regression": (
        LogisticRegression(max_iter=5000, solver="saga", random_state=RANDOM_STATE),
        # 10 C values so RandomizedSearchCV(n_iter=N_ITER) covers the full grid
        {"C": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 5.0, 10.0, 50.0, 100.0, 1000.0]},
    ),
    "random_forest": (
        RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100, 150, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
            "max_features": ["sqrt", 0.5],
        },
    ),
    "hist_gradient_boosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "max_iter": [100, 200],
            "max_depth": [None, 3, 5],
            "learning_rate": [0.05, 0.1, 0.2],
        },
    ),
}

In [4]:
CLEANED_PARAMS = {"valid_end_activities": sorted(VALID_END_ACTIVITIES)}

FEATURE_PARAMS = {
    **CLEANED_PARAMS,
    "min_prefix_length": MIN_PREFIX_LENGTH,
    "max_prefix_length": MAX_PREFIX_LENGTH,
    "feature_classes": [BasicControlFlowFeatures.__name__],
    "task": "classification",
}

ARTIFACT_PARAMS = {
    **FEATURE_PARAMS,
    "models": {
        name: {"class": type(est).__name__, "hp_grid": grid}
        for name, (est, grid) in MODEL_CONFIGS.items()
    },
    "n_iter": N_ITER,
    "cv_folds": CV_FOLDS,
    "random_state": RANDOM_STATE,
}

## tqdm_joblib helper

Patches joblib's batch-completion callback so that every completed parallel job advances a tqdm bar.
No additional packages required — `tqdm` and `joblib` are already project dependencies.

In [5]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old

## Target generator

In BPI 2017 the loan outcome is stored as the `Accepted` attribute on offer events — it is **not**
determined by the last activity of a trace (a case can end with `W_Validate application` regardless
of whether a loan was granted).

The target is derived by scanning the **full trace**: if any event has `Accepted == True`, the case
is a positive outcome (loan approved). Every prefix of the same case receives the same label.

The `TargetGenerator` protocol has no `col_idx` parameter. A module-level dict `_col_idx_store`
is populated by `my_splitter()` (which runs before feature extraction) and read by `loan_outcome` —
safe because pipeline stages run sequentially.

In [6]:
_col_idx_store: dict[str, int] = {}


def loan_outcome(trace: np.ndarray, start_idx: int, end_idx: int) -> int:
    accepted_idx = _col_idx_store["Accepted"]
    # Scan the full trace: any offer accepted → positive outcome
    return 1 if any(v is True for v in trace[:, accepted_idx]) else 0


def my_preprocessor(raw: RawData) -> EventLog:
    cleaned = clean_data(raw, valid_ends=VALID_END_ACTIVITIES)
    return cleaned.event_log


def my_splitter(log: EventLog) -> PreprocessedData:
    train_df, test_df = split_data(log)

    _col_idx_store.clear()
    _col_idx_store.update({c: i for i, c in enumerate(train_df.columns)})

    prefix_gen = sliding_window_factory(
        min_length=MIN_PREFIX_LENGTH,
        max_length=MAX_PREFIX_LENGTH,
    )

    return PreprocessedData(
        train_log=_build_trace_samples(train_df, prefix_gen),
        num_train_cases=len(train_df["case:concept:name"].unique()),
        test_log=_build_trace_samples(test_df, prefix_gen),
        num_test_cases=len(test_df["case:concept:name"].unique()),
        col_idx=_col_idx_store.copy(),
    )

## Stage 1 — Load & preprocess

In [7]:
dataset = Dataset()
raw = RawData(event_log=dataset.log)

cleaned = checkpoint(
    CHECKPOINT_DIR / "cleaned_log_clf.pkl",
    lambda: my_preprocessor(raw),
    params=CLEANED_PARAMS,
)
preprocessed = my_splitter(cleaned)

INFO:spi_time_series.data.dataset:Dataset found at G:\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
g:\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.
INFO:spi_time_series.pipeline.checkpointing:Loading checkpoint: ..\data\checkpoints\cleaned_log_clf__9ea2b74a.pkl
INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.


## Stage 2 — Feature extraction

`one_hot_encode_categorical` is left **off** (default). The bag-of-activities columns (`count_<activity>`)
already capture which activities appeared and how many times, so the information loss is acceptable for a baseline.

In [8]:
feature_extractor = extract_features_builder(
    [BasicControlFlowFeatures()],
    loan_outcome,
)

features = checkpoint(
    CHECKPOINT_DIR / "features_clf.pkl",
    lambda: feature_extractor(preprocessed),
    params=FEATURE_PARAMS,
)

print(f"Train shape: {features.X_train.shape}")
print(f"Test shape:  {features.X_test.shape}")
print(f"Class balance (train): {features.y_train.value_counts().to_dict()}")

INFO:spi_time_series.pipeline.checkpointing:Loading checkpoint: ..\data\checkpoints\features_clf__95edc9f3.pkl


Train shape: (822706, 33)
Test shape:  (227073, 33)
Class balance (train): {1: 594871, 0: 227835}


## Stages 3 & 4 — Hyperparameter search + fit

For each model: randomised CV search (`refit=False`) prints best params and CV AUC,
then `train()` fits every best estimator inside a full preprocessing pipeline.
All three models are wrapped in `_fit_artifact()` so the entire block is skipped when the artifact checkpoint exists.

In [9]:
SEARCH_SAMPLE_SIZE = 50_000  # subsample for CV search; final models train on full data

def _fit_artifact() -> ModelArtifact:
    from sklearn.pipeline import Pipeline as _SKPipeline
    from sklearn.model_selection import train_test_split as _tts
    search_pre = ColumnTransformer(
        [("num", _SKPipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), make_column_selector(dtype_include=np.number))],
        remainder="drop",
    )
    X_train_num = search_pre.fit_transform(features.X_train)

    # Subsample for faster CV search; 50K is enough to rank hyperparameters
    X_search, _, y_search, _ = _tts(
        X_train_num, features.y_train,
        train_size=SEARCH_SAMPLE_SIZE,
        stratify=features.y_train,
        random_state=RANDOM_STATE,
    )
    n_cv_jobs = N_ITER * CV_FOLDS
    print(f"Searching on {len(y_search):,} samples ({n_cv_jobs} CV fits total)")

    best_estimators = {}
    for name, (base_estimator, hp_grid) in MODEL_CONFIGS.items():
        print(f"\n--- {name} ---")
        search = RandomizedSearchCV(
            base_estimator,
            param_distributions=hp_grid,
            n_iter=N_ITER,
            cv=CV_FOLDS,
            scoring="roc_auc",
            refit=False,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        with tqdm_joblib(tqdm(desc=f"Search: {name}", total=n_cv_jobs)):
            search.fit(X_search, y_search)
        print(f"  best params: {search.best_params_}")
        print(f"  CV AUC:      {search.best_score_:.4f}")
        best_estimators[name] = clone(base_estimator).set_params(**search.best_params_)

    print("\nFitting final models on full training set...")
    return train(features, best_estimators)

In [10]:
artifact = checkpoint(CHECKPOINT_DIR / "artifact_clf.pkl", _fit_artifact, params=ARTIFACT_PARAMS)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\artifact_clf__87a637c1.pkl


Searching on 50,000 samples (30 CV fits total)

--- logistic_regression ---


Search: logistic_regression: 100%|██████████| 30/30 [01:27<00:00,  3.32s/it]

  best params: {'C': 1000.0}
  CV AUC:      0.5714

--- random_forest ---


  best params: {'n_estimators': 150, 'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 10}
  CV AUC:      0.5992

--- hist_gradient_boosting ---



Search: random_forest: 100%|██████████| 30/30 [00:09<00:00,  3.01it/s]








INFO:spi_time_series.models.trainer:Training model: logistic_regression


  best params: {'max_iter': 100, 'max_depth': None, 'learning_rate': 0.05}
  CV AUC:      0.5995

Fitting final models on full training set...



INFO:spi_time_series.models.trainer:Model logistic_regression trained.
INFO:spi_time_series.models.trainer:Training model: random_forest
INFO:spi_time_series.models.trainer:Model random_forest trained.
INFO:spi_time_series.models.trainer:Training model: hist_gradient_boosting
INFO:spi_time_series.models.trainer:Model hist_gradient_boosting trained.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\artifact_clf__87a637c1.pkl


## Stage 5 — Evaluate

In [11]:
_PREFIX_LENGTH_COL = "BasicControlFlowFeatures__prefix_length"
_logger = logging.getLogger(__name__)


def evaluate_classification(artifact: ModelArtifact, features_set) -> EvaluationReport:
    """Per-model, per-prefix-length classification metrics on the test set.

    Metrics: AUC (roc_auc_score), Brier Score, Accuracy.
    Uses predict_proba[:, 1] for AUC and Brier; predict for Accuracy.
    Requires BasicControlFlowFeatures__prefix_length in features.X_test.
    """
    X_test = features_set.X_test
    y_test = features_set.y_test

    if _PREFIX_LENGTH_COL not in X_test.columns:
        raise ValueError(
            f"Column '{_PREFIX_LENGTH_COL}' not found in X_test. "
            "BasicControlFlowFeatures must be included in the feature pipeline."
        )

    groups = X_test.groupby(_PREFIX_LENGTH_COL).groups
    prefix_lengths: list[int] = sorted(int(pl) for pl in groups)
    model_names: list[str] = list(artifact.models)
    all_metrics: dict = {}

    for model_name, pipeline in artifact.models.items():
        _logger.info("Evaluating model: %s", model_name)

        # Compute predictions once per model, then slice by group index
        y_prob = pd.Series(pipeline.predict_proba(X_test)[:, 1], index=X_test.index)
        y_pred = pd.Series(pipeline.predict(X_test), index=X_test.index)

        model_metrics: dict = {}
        for pl_val, group_idx in groups.items():
            y_true_g = y_test.loc[group_idx]
            y_prob_g = y_prob.loc[group_idx]
            y_pred_g = y_pred.loc[group_idx]

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    auc = float(roc_auc_score(y_true_g, y_prob_g))
                except ValueError:
                    auc = float("nan")
                model_metrics[int(pl_val)] = {
                    "auc": auc,
                    "brier": float(brier_score_loss(y_true_g, y_prob_g)),
                    "accuracy": float(accuracy_score(y_true_g, y_pred_g)),
                }

        all_metrics[model_name] = model_metrics
        _logger.info(
            "Model %s evaluated across %d prefix lengths.",
            model_name,
            len(prefix_lengths),
        )

    return EvaluationReport(
        metrics=all_metrics,
        model_names=model_names,
        prefix_lengths=prefix_lengths,
    )


report = evaluate_classification(artifact, features)

INFO:__main__:Evaluating model: logistic_regression
INFO:__main__:Model logistic_regression evaluated across 8 prefix lengths.
INFO:__main__:Evaluating model: random_forest
INFO:__main__:Model random_forest evaluated across 8 prefix lengths.
INFO:__main__:Evaluating model: hist_gradient_boosting
INFO:__main__:Model hist_gradient_boosting evaluated across 8 prefix lengths.


## Results

AUC, Brier Score, and Accuracy per model and prefix length.

In [12]:
rows = [
    {
        "model": model_name,
        "prefix_length": pl,
        "AUC": round(report.metrics[model_name][pl]["auc"], 4),
        "Brier Score": round(report.metrics[model_name][pl]["brier"], 4),
        "Accuracy": round(report.metrics[model_name][pl]["accuracy"], 4),
    }
    for model_name in sorted(report.model_names)
    for pl in report.prefix_lengths
]

pd.DataFrame(rows)

,model,prefix_length,AUC,Brier Score,Accuracy
0,hist_gradient_boosting,3,0.5592,0.1902,0.7412
1,hist_gradient_boosting,4,0.6235,0.1757,0.7680
2,hist_gradient_boosting,5,0.6299,0.1748,0.7677
3,hist_gradient_boosting,6,0.6312,0.1752,0.7672
4,hist_gradient_boosting,7,0.6287,0.1757,0.7671
5,hist_gradient_boosting,8,0.6236,0.1763,0.7669
6,hist_gradient_boosting,9,0.6236,0.1768,0.7656
7,hist_gradient_boosting,10,0.5726,0.1894,0.7432
8,logistic_regression,3,0.4740,0.1931,0.7412
9,logistic_regression,4,0.5792,0.1925,0.7412


### AUC comparison (pivot)

Rows = prefix length, columns = model.

In [13]:
df_auc = pd.DataFrame(
    {
        model_name: {
            pl: round(report.metrics[model_name][pl]["auc"], 4)
            for pl in report.prefix_lengths
        }
        for model_name in sorted(report.model_names)
    }
)
df_auc.index.name = "prefix_length"
df_auc

,hist_gradient_boosting,logistic_regression,random_forest
prefix_length,,,
3,0.5592,0.4740,0.5147
4,0.6235,0.5792,0.6239
5,0.6299,0.5865,0.6329
6,0.6312,0.5910,0.6316
7,0.6287,0.5777,0.6237
8,0.6236,0.5740,0.6171
9,0.6236,0.5754,0.6198
10,0.5726,0.5495,0.5691
